# Reporte de avance — Base de datos de víctimas y victimarios de extorsión

**Objetivo:** convertir noticias de extorsión en una base de datos con una fila por cada persona, grupo o empresa involucrada (víctima o victimario), usando un LLM que corre localmente.

**Metodología:** basada en Ortiz y Freitas (2025), *Automated News Scraping and AI-Powered Analysis for Municipal Crime Mapping*, reemplazando la API de Claude por un modelo local (Ollama, `qwen2.5:7b`).

**Estructura de la base de datos (relación uno a muchos):**
```
noticias (1) ──► eventos (1) ──► actores (N)
```

**Privacidad y derechos de autor:** no se almacenan nombres de personas ni se incluyen textos originales de noticias.

## 1. Librerías
Cargar las herramientas: `ollama` (modelo local), `pandas` (tablas) y `pydantic` (molde y validación de datos).

In [1]:
from typing import List, Literal, Optional

import ollama
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

MODELO = "qwen2.5:7b"

## 2. Esquema de datos
Define el "molde" que el modelo debe llenar. Un `Evento` (el caso de extorsión) contiene una **lista de `Actor`**.

- No existe un campo `nombre`: solo se guardan atributos (rol, rubro, sexo, edad, cantidad).
- `grupo_de_personas` + `cantidad` permite registrar casos como "12 detenidos" o "colectiveros" en una sola fila.

In [2]:
class Actor(BaseModel):
    rol: Literal["victima", "victimario"]
    tipo: Literal["persona", "grupo_de_personas", "empresa"]
    descripcion: str = Field(description="Quién es, SIN nombre propio: 'colectiveros informales', 'cabecilla de la banda'")
    rubro: Optional[str] = Field(None, description="Actividad de la víctima: transporte, construcción, inmobiliario, comercio…")
    sexo: Optional[Literal["masculino", "femenino"]] = None
    edad: Optional[int] = None
    cantidad: Optional[int] = Field(None, description="Número de personas si es un grupo")
    alias: Optional[str] = Field(None, description="SOLO el apodo de un victimario, si la noticia lo da")
    detenido: Optional[bool] = Field(None, description="Solo victimarios: true si fue capturado")


class Evento(BaseModel):
    es_extorsion: bool
    distrito: Optional[str] = None
    modalidad: Literal["cobro_de_cupo", "prestamo_gota_a_gota", "chalequeo", "otro", "no_especificado"]
    organizacion_mencionada: Optional[str] = None
    resumen: str = Field(description="Una frase, sin nombres de personas")
    actores: List[Actor]

## 3. Instrucciones para el modelo (prompt)
Reglas que guían la extracción. Esta versión corrige los errores de la primera prueba:
el alias se usaba para cosas que no eran apodos, no registraba a los detenidos y dejaba el rubro vacío.

In [3]:
PROMPT = """Eres un analista que convierte noticias policiales peruanas en datos estructurados.
Lee la noticia y completa el esquema JSON.

1. es_extorsion = true solo si alguien exige dinero bajo amenaza.
2. ACTORES: una entrada por cada víctima y cada victimario.
   - Un grupo ("colectiveros", "12 detenidos") es UNA entrada tipo "grupo_de_personas" con su cantidad.
   - Un victimario con rol especial (cabecilla, lugarteniente) tiene su propia entrada.
   - Si hay víctimas de distintos rubros, crea una entrada por cada rubro.
3. RUBRO: indica la actividad de cada víctima (transporte, construcción, inmobiliario, comercio...).
4. DETENIDOS: si la noticia dice que hubo capturas, marca detenido = true en los victimarios capturados
   e indica la cantidad total de detenidos.
5. ALIAS: solo el apodo de una persona (ej. una palabra entre comillas). NO pongas ahí el nombre
   de la banda, el cargo ni descripciones. El nombre de la banda va en organizacion_mencionada.
6. NUNCA escribas nombres propios de personas. Usa su rol: "cabecilla de la banda", "dueño de obra".
7. Si un dato no aparece, usa null. No inventes. Responde solo con el JSON."""

## 4. Función de extracción
Envía la noticia al modelo local, valida la respuesta con el esquema y reintenta si el formato no es válido.

In [4]:
def extraer(noticia: dict, intentos: int = 2) -> Optional[Evento]:
    mensaje = f"Fecha de publicación: {noticia['fecha_publicacion']}\n\nNoticia:\n{noticia['texto']}"
    for i in range(intentos):
        r = ollama.chat(model=MODELO,
                        messages=[{"role": "system", "content": PROMPT},
                                  {"role": "user", "content": mensaje}],
                        format=Evento.model_json_schema(),
                        options={"temperature": 0, "num_ctx": 8192})
        try:
            return Evento.model_validate_json(r.message.content)
        except ValidationError:
            print(f"  intento {i + 1}: respuesta inválida, reintentando…")
    return None

## 5. Noticia de prueba
**No es el texto original.** Es un resumen propio de los hechos reportados por RPP el 12/10/2026 (https://rpp.pe/lima/actualidad/lima-desarticulan-presunta-banda-criminal-que-exigia-cupos-semanales-de-s-120-a-colectiveros-y-pagos-de-hasta-s-50-000-a-empresarios-noticia-1712178?ref=rpp), redactado sin nombres de personas para respetar los derechos de autor y la privacidad.

In [5]:
noticias = [
    {"id_noticia": "rpp-2026-10-12", "medio": "RPP", "fecha_publicacion": "2026-10-12",
     "url": "https://rpp.pe/lima/actualidad/lima-desarticulan-presunta-banda-criminal-que-exigia-cupos-semanales-de-s-120-a-colectiveros-y-pagos-de-hasta-s-50-000-a-empresarios-noticia-1712178",
     "texto": (
        "Según el Ministerio Público, en la madrugada del lunes se realizaron operativos en el Rímac y "
        "San Juan de Lurigancho contra una presunta organización criminal denominada 'Los injertos del "
        "Rímac - Nueva Generación'. Fueron detenidas 12 personas, entre ellas el presunto cabecilla y su "
        "lugarteniente. La banda habría cobrado a colectiveros informales una inscripción de entre 150 y "
        "250 soles por vehículo y una cuota semanal de 120 soles, de forma presencial o mediante billeteras "
        "digitales. Además, exigía pagos de entre 20 000 y 50 000 soles a propietarios de inmuebles, dueños "
        "de obras de construcción y empresas del Rímac, bajo amenazas contra su vida y la de sus familias."
     )},
]
print(len(noticias), "noticia(s) cargada(s)")

1 noticia(s) cargada(s)


## 6. Construcción de las tablas
Cada noticia genera **1 fila en `eventos`** y **N filas en `actores`**, unidas por `id_evento`.

In [6]:
filas_eventos, filas_actores = [], []

for i, n in enumerate(noticias, start=1):
    print(f"Procesando {n['id_noticia']}…")
    ev = extraer(n)
    if ev is None:
        print("  ✘ no se pudo extraer")
        continue
    id_evento = f"ev-{i:03d}"
    datos = ev.model_dump()
    actores = datos.pop("actores")
    filas_eventos.append({"id_evento": id_evento, "id_noticia": n["id_noticia"],
                          "medio": n["medio"], "url": n["url"], **datos})
    for j, a in enumerate(actores, start=1):
        filas_actores.append({"id_actor": f"{id_evento}-a{j:02d}", "id_evento": id_evento, **a})

df_eventos = pd.DataFrame(filas_eventos)
df_actores = pd.DataFrame(filas_actores)
print(f"{len(df_eventos)} evento(s), {len(df_actores)} actor(es)")
df_eventos

Procesando rpp-2026-10-12…
1 evento(s), 7 actor(es)


,id_evento,id_noticia,medio,url,es_extorsion,distrito,modalidad,organizacion_mencionada,resumen
0,ev-001,rpp-2026-10-12,RPP,https://rpp.pe/lima/actualidad/lima-desarticul...,True,None,cobro_de_cupo,Los injertos del Rímac - Nueva Generación,Operativos contra 'Los injertos del Rímac - Nu...


### Tabla de actores

In [7]:
df_actores

,id_actor,id_evento,rol,tipo,descripcion,rubro,sexo,edad,cantidad,alias,detenido
0,ev-001-a01,ev-001,victimario,grupo_de_personas,Los injertos del Rímac - Nueva Generación,None,None,None,12.0,None,True
1,ev-001-a02,ev-001,victimario,persona,cabecilla de la banda,None,None,None,NaN,None,None
2,ev-001-a03,ev-001,victimario,persona,lugarteniente,None,None,None,NaN,None,None
3,ev-001-a04,ev-001,victima,persona,colectivero informal,None,None,None,NaN,None,None
4,ev-001-a05,ev-001,victima,persona,propietario de inmueble,None,None,None,NaN,None,None
5,ev-001-a06,ev-001,victima,persona,dueño de obra de construcción,None,None,None,NaN,None,None
6,ev-001-a07,ev-001,victima,persona,empresario,None,None,None,NaN,None,None


## 7. Exportación con protección de datos
Antes de guardar, los alias se reemplazan por `[omitido]`: un alias junto a la fecha, el distrito y la condición de detenido podría identificar a una persona real.

In [8]:
publico = df_actores.copy()
publico["alias"] = publico["alias"].where(publico["alias"].isna(), "[omitido]")

df_eventos.to_csv("eventos.csv", index=False, encoding="utf-8-sig")
publico.to_csv("actores.csv", index=False, encoding="utf-8-sig")
print("Guardado: eventos.csv y actores.csv")
publico

Guardado: eventos.csv y actores.csv


,id_actor,id_evento,rol,tipo,descripcion,rubro,sexo,edad,cantidad,alias,detenido
0,ev-001-a01,ev-001,victimario,grupo_de_personas,Los injertos del Rímac - Nueva Generación,None,None,None,12.0,None,True
1,ev-001-a02,ev-001,victimario,persona,cabecilla de la banda,None,None,None,NaN,None,None
2,ev-001-a03,ev-001,victimario,persona,lugarteniente,None,None,None,NaN,None,None
3,ev-001-a04,ev-001,victima,persona,colectivero informal,None,None,None,NaN,None,None
4,ev-001-a05,ev-001,victima,persona,propietario de inmueble,None,None,None,NaN,None,None
5,ev-001-a06,ev-001,victima,persona,dueño de obra de construcción,None,None,None,NaN,None,None
6,ev-001-a07,ev-001,victima,persona,empresario,None,None,None,NaN,None,None


## 8. Resumen

Con una noticia de prueba, el modelo generó 1 evento y 7 actores (3 victimarios y 4 víctimas).

**Funciona (mejoras respecto a la primera prueba):**
- Separa víctimas y victimarios sin escribir nombres propios.
- Registra a los detenidos como grupo (cantidad = 12, detenido = true).
- El campo alias ya no se usa para cargos ni descripciones.
- Crea una entrada por cada tipo de víctima (colectiveros, propietarios de inmuebles, dueños de obra, empresarios).

**Por mejorar:**
- El campo `rubro` queda vacío.
- Las víctimas en plural se clasifican como `persona`.
- El cabecilla y el lugarteniente no se marcan como detenidos, aunque estaban entre los 12 capturados.

**Próximos pasos:**
- Ajustar el prompt con ejemplos para rubro y tipo de actor.
- Probar con 10 noticias y medir la precisión por campo.
- Conectar con el scraping de noticias.